In [ ]:
# !pip install kiwipiepy[full]

zsh:1: no matches found: kiwipiepy[full]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from kiwipiepy import Kiwi
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.api.types import CategoricalDtype

kiwi = Kiwi()

In [2]:
import os
print(os.getcwd())

/Users/hs/SPARTA_PROJECT_4TH_E_COMMERCE/김혜수


In [3]:
df_alo_reviews = pd.read_csv('./data/alo_reviews_final.csv')
df_lululemon_reviews = pd.read_csv('./data/lululemon_reviews_master_20260417.csv')
df_lululemon_reviews_musinsa = pd.read_csv('./data/musinsa_lululemon_reviews.csv')
df_xexymix_reviews = pd.read_csv('./data/xexymix_raw_review_final.csv')
df_xexymix_reviews_musinsa = pd.read_csv('./data/musinsa_xexymix_reviews.csv')
df_naver_blog = pd.read_csv('./data/naver_blog.csv')

## 룰루레몬 리뷰

## 룰루레몬 무신사 리뷰

In [205]:
df_lululemon_reviews_musinsa.info()

<class 'pandas.DataFrame'>
Index: 1581 entries, 0 to 1943
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   collect_date       1581 non-null   datetime64[us]
 1   platform           0 non-null      category      
 2   review_id          1581 non-null   int64         
 3   product_id         1581 non-null   int64         
 4   product_name       1581 non-null   str           
 5   review_date        1581 non-null   datetime64[us]
 6   year               0 non-null      object        
 7   month              0 non-null      object        
 8   content            0 non-null      object        
 9   rating             1581 non-null   int64         
 10  helpful_count      1581 non-null   int64         
 11  has_image          1581 non-null   int64         
 12  purchase_option    1581 non-null   str           
 13  user_height        1257 non-null   float64       
 14  user_weight        1257 

In [187]:
df_lululemon_reviews_musinsa.shape

(1581, 18)

### 컬럼명 변경

In [206]:
df_lululemon_reviews_musinsa = df_lululemon_reviews_musinsa.rename(columns={
    'review_no' : 'review_id',
    'review_text' : 'content',
    'helpful' : 'helpful_count',
    'option' : 'purchase_option'
})

### 결측치 % 확인

In [189]:
(df_lululemon_reviews_musinsa.isna().mean() * 100).sort_values(ascending=False)

product_url          100.000000
year                 100.000000
month                100.000000
review_body          100.000000
platform             100.000000
user_weight_group     20.493359
user_height_group     20.493359
user_weight           20.493359
user_height           20.493359
has_image              0.000000
purchase_option        0.000000
collect_date           0.000000
helpful_count          0.000000
review_date            0.000000
product_name           0.000000
product_id             0.000000
review_id              0.000000
rating                 0.000000
dtype: float64

### 형 변환(datetime)

In [190]:
df_lululemon_reviews_musinsa['collect_date'] = pd.to_datetime(df_lululemon_reviews_musinsa['collect_date'])
df_lululemon_reviews_musinsa['review_date'] = pd.to_datetime(df_lululemon_reviews_musinsa['review_date'])

### 형 변환(int)

In [191]:
df_lululemon_reviews_musinsa['has_image'] = df_lululemon_reviews_musinsa['has_image'].astype(int)

In [192]:
df_lululemon_reviews_musinsa['has_image']

0       1
1       0
2       1
3       1
4       0
       ..
1939    0
1940    1
1941    1
1942    0
1943    0
Name: has_image, Length: 1581, dtype: int64

### 고객 키 그룹화

In [193]:
df_lululemon_reviews_musinsa['user_height'] = pd.to_numeric(df_lululemon_reviews_musinsa['user_height'], errors='coerce')

# 구간 경계 정의
bins = [0, 139, 144, 149, 154, 159, 164, 169, 174, 179, 184, 189, float('inf')]

# 라벨 정의
user_height_group = [
    '139cm 이하',
    '140~144cm',
    '145~149cm',
    '150~154cm',
    '155~159cm',
    '160~164cm',
    '165~169cm',
    '170~174cm',
    '175~179cm',
    '180~184cm',
    '185~189cm',
    '190cm 이상'
]

df_lululemon_reviews_musinsa['user_height_group'] = pd.cut(df_lululemon_reviews_musinsa['user_height'], bins=bins, labels=user_height_group, right=False)

In [194]:
df_lululemon_reviews_musinsa['user_height_group'].value_counts()

user_height_group
165~169cm    293
160~164cm    245
175~179cm    233
170~174cm    207
180~184cm    113
155~159cm     95
185~189cm     41
150~154cm     26
139cm 이하       3
190cm 이상       1
140~144cm      0
145~149cm      0
Name: count, dtype: int64

### 고객 몸무게 그룹화

In [195]:
# 구간 경계 정의 (right=False로 아래 경계 포함, 위 경계 미포함)
bins = [float('-inf'), 44, 49, 54, 59, 64, 69, 74, 79, 84, 89, float('inf')]

# 라벨 정의 (주어진 순서대로)
user_weight_group = [
    '44kg 이하',
    '45-49kg',
    '50-54kg',
    '55-59kg',
    '60-64kg',
    '65-69kg',
    '70-74kg',
    '75-79kg',
    '80-84kg',
    '85-89kg',
    '90kg 이상'
]

# 데이터프레임 가정
# df = pd.DataFrame({'user_height': [150, 160, 170, 180]})

df_lululemon_reviews_musinsa['user_weight_group'] = pd.cut(df_lululemon_reviews_musinsa['user_weight'], bins=bins, labels=user_weight_group, right=False)


In [196]:
df_lululemon_reviews_musinsa['user_weight_group'].value_counts()

user_weight_group
50-54kg    206
55-59kg    190
70-74kg    184
60-64kg    129
75-79kg    125
65-69kg    112
45-49kg    109
80-84kg     77
85-89kg     60
44kg 이하     36
90kg 이상     29
Name: count, dtype: int64

### 형 변환(category)

In [197]:
df_lululemon_reviews_musinsa['platform'] = df_lululemon_reviews_musinsa['platform'].astype('category')

### 없는 컬럼 추가

In [204]:
# 없는 컬럼 추가하기
required_columns = [
    'collect_date', 'platform', 'review_id', 'product_id', 'product_name',
    'review_date', 'year', 'month', 'content', 'rating', 'helpful_count',
    'has_image', 'purchase_option', 'user_height', 'user_weight',
    'user_height_group', 'user_weight_group', 'product_url'
]

for col in required_columns:
    if col not in df_lululemon_reviews_musinsa.columns:
        df_lululemon_reviews_musinsa[col] = None

# 컬럼 순서 정리
df_lululemon_reviews_musinsa = df_lululemon_reviews_musinsa[required_columns]

In [199]:
df_lululemon_reviews_musinsa['user_weight_group']

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
         ...   
1939    65-69kg
1940    55-59kg
1941    75-79kg
1942    70-74kg
1943    75-79kg
Name: user_weight_group, Length: 1581, dtype: category
Categories (11, str): ['44kg 이하' < '45-49kg' < '50-54kg' < '55-59kg' ... '75-79kg' < '80-84kg' < '85-89kg' < '90kg 이상']

### ```review_id``` 중복 제거

In [200]:
df_lululemon_reviews_musinsa = df_lululemon_reviews_musinsa.drop_duplicates(subset=['review_id'])

### 전처리

In [207]:
df_lululemon_reviews_musinsa['content'] = (
    df_lululemon_reviews_musinsa['content']
    .fillna('')  # 결측치 처리
    .apply(lambda x: re.sub(r'\s+', ' ', str(x)))  # 모든 공백 통합 (\n, \t 포함)
    .str.replace(r'[^가-힣a-zA-Z0-9\s]', '', regex=True)  # 특수문자 제거
    .str.strip()  # 앞뒤 공백 제거
)

### 2024 이후 데이터 저장

In [ ]:
# 데이터 분리
lululemon_musinsa_reviews_final_s2024 = df_lululemon_reviews_musinsa[
    df_lululemon_reviews_musinsa['review_date'] >= '2024-01-01'
].copy()

lululemon_musinsa_reviews_final = df_lululemon_reviews_musinsa.copy()

# 삭제할 컬럼
cols_to_drop = ['review_type', 'author', 'comment_count', 'image_urls', 'survey_size', 'survey_width', 'survey_comfort', 'survey_instep']

# 컬럼 제거
lululemon_musinsa_reviews_final_s2024 = lululemon_musinsa_reviews_final_s2024.drop(columns=cols_to_drop, errors='ignore')
lululemon_musinsa_reviews_final = lululemon_musinsa_reviews_final.drop(columns=cols_to_drop, errors='ignore')

# 저장
lululemon_musinsa_reviews_final_s2024.to_csv('data_pre/lululemon_musinsa_reviews_final_s2024.csv', index=False, encoding='utf-8-sig')
lululemon_musinsa_reviews_final.to_csv('data_pre/lululemon_musinsa_reviews_final.csv', index=False, encoding='utf-8-sig')

## 젝시믹스 리뷰

In [ ]:
df_xexymix_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 2364102 entries, 0 to 2364101
Data columns (total 16 columns):
 #   Column             Dtype         
---  ------             -----         
 0   collect_date       datetime64[us]
 1   review_date        datetime64[us]
 2   platform           category      
 3   brand              str           
 4   search_keyword     str           
 5   content            str           
 6   product_url        str           
 7   rating             int64         
 8   helpful_count      int64         
 9   purchased_option   category      
 10  has_image          int64         
 11  user_height        float64       
 12  user_weight        float64       
 13  user_size          category      
 14  user_height_group  category      
 15  user_weight_group  category      
dtypes: category(5), datetime64[us](2), float64(2), int64(3), str(4)
memory usage: 217.0 MB


In [277]:
df_xexymix_reviews.shape

(2364102, 14)

In [294]:
df_xexymix_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 2364102 entries, 0 to 2364101
Data columns (total 16 columns):
 #   Column             Dtype         
---  ------             -----         
 0   collect_date       datetime64[us]
 1   review_date        datetime64[us]
 2   platform           category      
 3   brand              str           
 4   search_keyword     str           
 5   content            str           
 6   product_url        str           
 7   rating             int64         
 8   helpful_count      int64         
 9   purchased_option   category      
 10  has_image          int64         
 11  user_height        float64       
 12  user_weight        float64       
 13  user_size          category      
 14  user_height_group  category      
 15  user_weight_group  category      
dtypes: category(5), datetime64[us](2), float64(2), int64(3), str(4)
memory usage: 217.0 MB


### 컬럼명 변경

In [279]:
df_xexymix_reviews = df_xexymix_reviews.rename(columns={
    'post_date' : 'review_date',
    'source_url' : 'product_url'
})

### 형 변환(datetime)

In [280]:
df_xexymix_reviews['review_date'] = pd.to_datetime(df_xexymix_reviews['review_date'])
df_xexymix_reviews['collect_date'] = pd.to_datetime(df_xexymix_reviews['collect_date'])

### 형 변환(category)

In [281]:
df_xexymix_reviews['platform'] = df_xexymix_reviews['platform'].astype('category')
df_xexymix_reviews['purchased_option'] = df_xexymix_reviews['purchased_option'].astype('category')
df_xexymix_reviews['user_size'] = df_xexymix_reviews['user_size'].astype('category')

### 결측치 % 확인

In [282]:
(df_xexymix_reviews.isna().mean() * 100).sort_values(ascending=False)

search_keyword      100.000000
user_size            17.906165
user_weight          13.323241
user_height          10.445065
collect_date          0.000000
review_date           0.000000
platform              0.000000
brand                 0.000000
content               0.000000
product_url           0.000000
rating                0.000000
helpful_count         0.000000
purchased_option      0.000000
has_image             0.000000
dtype: float64

### 컬럼만 남기고 데이터 제거(컬럼만 유지)

In [283]:
df_xexymix_reviews['search_keyword'] = ''

In [284]:
df_xexymix_reviews['search_keyword']

0           
1           
2           
3           
4           
          ..
2364097     
2364098     
2364099     
2364100     
2364101     
Name: search_keyword, Length: 2364102, dtype: str

### 이상치 확인

In [285]:
df_xexymix_reviews['helpful_count'].describe()

count    2.364102e+06
mean     1.536748e-01
std      2.744528e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.110000e+02
Name: helpful_count, dtype: float64

In [286]:
df_xexymix_reviews['has_image'].value_counts()

has_image
1    2364102
Name: count, dtype: int64

### 고객 키 그룹화

In [287]:
df_xexymix_reviews['user_height'] = pd.to_numeric(df_xexymix_reviews['user_height'], errors='coerce')

# 구간 경계 정의
bins = [0, 139, 144, 149, 154, 159, 164, 169, 174, 179, 184, 189, float('inf')]

# 라벨 정의
user_height_group = [
    '139cm 이하',
    '140~144cm',
    '145~149cm',
    '150~154cm',
    '155~159cm',
    '160~164cm',
    '165~169cm',
    '170~174cm',
    '175~179cm',
    '180~184cm',
    '185~189cm',
    '190cm 이상'
]

df_xexymix_reviews['user_height_group'] = pd.cut(df_xexymix_reviews['user_height'], bins=bins, labels=user_height_group, right=False)

In [288]:
df_xexymix_reviews['user_height_group'].value_counts()

user_height_group
160~164cm    701185
165~169cm    504017
155~159cm    392435
170~174cm    223250
175~179cm    121961
150~154cm     66980
180~184cm     56722
185~189cm     13630
139cm 이하       3555
145~149cm      2643
190cm 이상       2155
140~144cm       645
Name: count, dtype: int64

### 고객 몸무게 그룹화

In [289]:
df_xexymix_reviews['user_weight'] = pd.to_numeric(df_xexymix_reviews['user_weight'], errors='coerce')

In [290]:
# 구간 경계 정의
bins = [0, 44, 49, 54, 59, 64, 69, 74, 79, 84, 89, float('inf')]

# 라벨 정의
user_weight_group = [
    '44kg 이하',
    '45~49kg',
    '50~54kg',
    '55~59kg',
    '60~64kg',
    '65~69kg',
    '70~74kg',
    '75~79kg',
    '80~84kg',
    '85~89kg',
    '90kg 이상'
]

df_xexymix_reviews['user_weight_group'] = pd.cut(df_xexymix_reviews['user_weight'], bins=bins, labels=user_weight_group, right=False)

In [291]:
df_xexymix_reviews['user_weight_group'].value_counts()

user_weight_group
50~54kg    553633
55~59kg    441703
45~49kg    343202
60~64kg    257635
65~69kg    144090
70~74kg    101162
75~79kg     72312
80~84kg     56889
90kg 이상     44481
85~89kg     32797
44kg 이하      1223
Name: count, dtype: int64

### 전처리

In [292]:
df_xexymix_reviews['content'] = (
    df_xexymix_reviews['content']
    .fillna('')  # 결측치 처리
    .apply(lambda x: re.sub(r'\s+', ' ', str(x)))  # 모든 공백 통합 (\n, \t 포함)
    .str.replace(r'[^가-힣a-zA-Z0-9\s]', '', regex=True)  # 특수문자 제거
    .str.strip()  # 앞뒤 공백 제거
)

### 2024년 이후 데이터 저장

In [207]:
# 데이터 분리
xexymix_homepage_review_final_s2024 = df_xexymix_reviews[
    df_xexymix_reviews['review_date'] >= '2024-01-01'
].copy()

xexymix_homepage_review_final = df_xexymix_reviews.copy()

# 삭제할 컬럼
cols_to_drop = ['brand']

# 컬럼 제거
xexymix_homepage_review_final_s2024 = xexymix_homepage_review_final_s2024.drop(columns=cols_to_drop, errors='ignore')
xexymix_homepage_review_final = xexymix_homepage_review_final.drop(columns=cols_to_drop, errors='ignore')

# 저장
xexymix_homepage_review_final_s2024.to_csv('data_pre/xexymix_homepage_review_final_s2024.csv', index=False, encoding='utf-8-sig')
xexymix_homepage_review_final.to_csv('data_pre/xexymix_homepage_review_final.csv', index=False, encoding='utf-8-sig')

## 젝시믹스 무신사 리뷰

In [271]:
df_xexymix_reviews_musinsa.info()

<class 'pandas.DataFrame'>
RangeIndex: 15086 entries, 0 to 15085
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   collect_date       15086 non-null  str     
 1   platform           0 non-null      category
 2   review_id          15086 non-null  int64   
 3   product_id         15086 non-null  int64   
 4   product_name       15086 non-null  str     
 5   review_date        15086 non-null  str     
 6   year               0 non-null      object  
 7   month              0 non-null      object  
 8   review_body        0 non-null      object  
 9   rating             15086 non-null  int64   
 10  helpful_count      15086 non-null  int64   
 11  has_image          15086 non-null  bool    
 12  purchase_option    15086 non-null  category
 13  user_height        15086 non-null  int64   
 14  user_weight        15086 non-null  int64   
 15  user_height_group  15086 non-null  category
 16  user_weight_gro

In [259]:
df_xexymix_reviews_musinsa.shape

(15086, 21)

### 컬럼명 변경

In [260]:
df_xexymix_reviews_musinsa = df_xexymix_reviews_musinsa.rename(columns={
    'review_no' : 'review_id',
    'review_text' : 'content',
    'option' : 'purchase_option',
    'url' : 'product_url',
    'helpful' : 'helpful_count'
})

### 고객 키 그룹화

In [261]:
df_xexymix_reviews_musinsa['user_height'] = pd.to_numeric(df_xexymix_reviews_musinsa['user_height'], errors='coerce')

In [262]:
# 구간 경계 정의
bins = [0, 139, 144, 149, 154, 159, 164, 169, 174, 179, 184, 189, float('inf')]

# 라벨 정의
user_height_group = [
    '139cm 이하',
    '140~144cm',
    '145~149cm',
    '150~154cm',
    '155~159cm',
    '160~164cm',
    '165~169cm',
    '170~174cm',
    '175~179cm',
    '180~184cm',
    '185~189cm',
    '190cm 이상'
]

df_xexymix_reviews_musinsa['user_height_group'] = pd.cut(df_xexymix_reviews_musinsa['user_height'], bins=bins, labels=user_height_group, right=False)

In [263]:
df_xexymix_reviews_musinsa['user_height_group'].value_counts()

user_height_group
160~164cm    3938
165~169cm    3446
155~159cm    1956
170~174cm    1775
139cm 이하     1716
175~179cm    1153
180~184cm     562
150~154cm     294
185~189cm     189
190cm 이상       36
140~144cm      11
145~149cm      10
Name: count, dtype: int64

### 고객 몸무게 그룹화

In [264]:
# 구간 경계 정의 (right=False로 아래 경계 포함, 위 경계 미포함)
bins = [float('-inf'), 44, 49, 54, 59, 64, 69, 74, 79, 84, 89, float('inf')]

# 라벨 정의 (주어진 순서대로)
user_weight_group = [
    '44kg 이하',
    '45-49kg',
    '50-54kg',
    '55-59kg',
    '60-64kg',
    '65-69kg',
    '70-74kg',
    '75-79kg',
    '80-84kg',
    '85-89kg',
    '90kg 이상'
]

# 데이터프레임 가정
# df = pd.DataFrame({'user_height': [150, 160, 170, 180]})

df_xexymix_reviews_musinsa['user_weight_group'] = pd.cut(df_xexymix_reviews_musinsa['user_weight'], bins=bins, labels=user_weight_group, right=False)


### 없는 컬럼 추가

In [265]:
# 없는 컬럼 추가하기
required_columns = [
    'collect_date', 'platform', 'review_id', 'product_id', 'product_name',
    'review_date', 'year', 'month', 'review_body', 'rating', 'helpful_count',
    'has_image', 'purchase_option', 'user_height', 'user_weight',
    'user_height_group', 'user_weight_group', 'product_url'
]

for col in required_columns:
    if col not in df_xexymix_reviews_musinsa.columns:
        df_xexymix_reviews_musinsa[col] = None

# 컬럼 순서 정리
df_xexymix_reviews_musinsa = df_xexymix_reviews_musinsa[required_columns]

### 형 변환(category)

In [266]:
df_xexymix_reviews_musinsa['platform'] = df_xexymix_reviews_musinsa['platform'].astype('category')
df_xexymix_reviews_musinsa['user_height_group'] = df_xexymix_reviews_musinsa['user_height_group'].astype('category')
df_xexymix_reviews_musinsa['purchase_option'] = df_xexymix_reviews_musinsa['purchase_option'].astype('category')

In [267]:
df_xexymix_reviews_musinsa['user_weight_group'].value_counts()

user_weight_group
50-54kg    3277
55-59kg    3060
44kg 이하    2124
45-49kg    1826
60-64kg    1612
65-69kg     838
70-74kg     744
75-79kg     586
80-84kg     425
90kg 이상     299
85-89kg     295
Name: count, dtype: int64

In [272]:
df_xexymix_reviews_musinsa['product_name'].value_counts()

product_name
젤라 인텐션 바이커 쇼츠 3부 블랙 XP9172F               2530
젤라 인텐션 부츠컷 팬츠 블랙 XWFLG02H3_XWFLG06J1      1296
젤라 인텐션 부츠컷 팬츠 제트차콜 XWFLG02H3_XWFLG06J1     504
[남녀공용] 서포트핏 무릎 보호대 XAUZZ25J3               358
[2PACK] V업 3D 플러스 레깅스 XP9156T              322
                                          ... 
RX 쿨라이트 백메쉬 슬리브리스 XFL2TE1012                 1
NX 브이넥 인패드 크롭 숏슬리브 XFL2BC1511                1
루프 스트랩 리버서블 버킷백 XUL5AB1705                   1
NX 맨즈 스포티 스웨트 후드 집업 XML1TC1106               1
아이스페더 라이트 2.0 언밸런스 루즈핏 숏슬리브 XFK1TS1003       1
Name: count, Length: 867, dtype: int64

In [273]:
(df_xexymix_reviews_musinsa.isna().mean() * 100).sort_values(ascending=False)

year                 100.0
month                100.0
review_body          100.0
platform             100.0
collect_date           0.0
has_image              0.0
user_weight_group      0.0
user_height_group      0.0
user_weight            0.0
user_height            0.0
purchase_option        0.0
rating                 0.0
helpful_count          0.0
review_date            0.0
product_name           0.0
product_id             0.0
review_id              0.0
product_url            0.0
dtype: float64

In [274]:
df_xexymix_reviews_musinsa['user_weight']

0        62
1        62
2        53
3        55
4        50
         ..
15081    97
15082    90
15083    57
15084    78
15085    78
Name: user_weight, Length: 15086, dtype: int64

### 형 변환(datetime)

In [108]:
df_xexymix_reviews_musinsa['collect_date'] = pd.to_datetime(df_xexymix_reviews_musinsa['collect_date'])
df_xexymix_reviews_musinsa['review_date'] = pd.to_datetime(df_xexymix_reviews_musinsa['review_date'])

### 형 변환(int)

In [231]:
df_xexymix_reviews_musinsa['has_image'] = df_xexymix_reviews_musinsa['has_image'].astype(int)

In [111]:
df_xexymix_reviews_musinsa['user_height_group'] = df_xexymix_reviews_musinsa['user_height_group'].astype(str)
df_xexymix_reviews_musinsa['user_weight_group'] = df_xexymix_reviews_musinsa['user_weight_group'].astype(str)

In [112]:
df_xexymix_reviews_musinsa['user_height'] = df_xexymix_reviews_musinsa['user_height'].astype(float)
df_xexymix_reviews_musinsa['user_weight'] = df_xexymix_reviews_musinsa['user_weight'].astype(float)

### 형 변환(category)

In [223]:
df_xexymix_reviews_musinsa['purchased_option'] = df_xexymix_reviews_musinsa['purchased_option'].astype('category')

KeyError: 'purchased_option'

### 결측치 % 확인

In [132]:
(df_xexymix_reviews_musinsa.isna().mean() * 100).sort_values(ascending=False)

survey_instep        100.000000
survey_width          99.960228
survey_size           99.661938
survey_comfort        99.151531
image_urls            38.976535
user_height           11.036723
user_weight           11.036723
collect_date           0.000000
comment_count          0.000000
user_height_group      0.000000
url                    0.000000
has_image              0.000000
option                 0.000000
helpful                0.000000
review_date            0.000000
author                 0.000000
review_text            0.000000
rating                 0.000000
review_type            0.000000
review_no              0.000000
product_name           0.000000
product_id             0.000000
user_weight_group      0.000000
dtype: float64

### ```0```값 -> nan 처리

In [125]:
cols = ['user_height', 'user_weight']

df_xexymix_reviews_musinsa[cols] = df_xexymix_reviews_musinsa[cols].replace(0, np.nan)

### ```review_id``` 중복 제거

In [129]:
df_xexymix_reviews_musinsa = df_xexymix_reviews_musinsa.drop_duplicates(subset=['review_id'])

### 이상치 확인

In [133]:
df_xexymix_reviews_musinsa['user_height'].describe()

count    13421.000000
mean       164.963788
std          9.436249
min         40.000000
25%        160.000000
50%        164.000000
75%        170.000000
max        220.000000
Name: user_height, dtype: float64

In [144]:
zero_count = df_xexymix_reviews_musinsa[df_xexymix_reviews_musinsa['user_height'] == 0].shape[0]
print(f"user_height가 0인 데이터 개수: {zero_count}")

user_height가 0인 데이터 개수: 1619


In [146]:
zero_count = df_xexymix_reviews_musinsa[df_xexymix_reviews_musinsa['user_weight'] == 0].shape[0]
print(f"user_weight가 0인 데이터 개수: {zero_count}")

user_weight가 0인 데이터 개수: 1619


### 전처리

In [114]:
df_xexymix_reviews_musinsa['content'] = (
    df_xexymix_reviews['content']
    .fillna('')  # 결측치 처리
    .apply(lambda x: re.sub(r'\s+', ' ', str(x)))  # 모든 공백 통합 (\n, \t 포함)
    .str.replace(r'[^가-힣a-zA-Z0-9\s]', '', regex=True)  # 특수문자 제거
    .str.strip()  # 앞뒤 공백 제거
)

### 2024년 이후 데이터 저장

In [208]:
# 데이터 분리
xexymix_musinsa_reviews_final_s2024 = df_xexymix_reviews_musinsa[
    df_xexymix_reviews_musinsa['review_date'] >= '2024-01-01'
].copy()

xexymix_musinsa_reviews_final = df_xexymix_reviews_musinsa.copy()

# 삭제할 컬럼
cols_to_drop = ['review_type', 'author', 'comment_count', 'image_urls', 'survey_size', 'survey_width', 'survey_comfort', 'survey_instep']

# 컬럼 제거
xexymix_musinsa_reviews_final_s2024 = xexymix_musinsa_reviews_final_s2024.drop(columns=cols_to_drop, errors='ignore')
xexymix_musinsa_reviews_final = xexymix_musinsa_reviews_final.drop(columns=cols_to_drop, errors='ignore')

# 저장
xexymix_musinsa_reviews_final_s2024.to_csv('data_pre/xexymix_musinsa_reviews_final_s2024.csv', index=False, encoding='utf-8-sig')
xexymix_musinsa_reviews_final.to_csv('data_pre/xexymix_musinsa_reviews_final.csv', index=False, encoding='utf-8-sig')

## 알로 리뷰

In [87]:
df_alo_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 42902 entries, 0 to 42901
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   review_id         42902 non-null  int64
 1   collect_date      42902 non-null  str  
 2   platform          42902 non-null  str  
 3   brand             42902 non-null  str  
 4   search_keyword    42902 non-null  str  
 5   content           42887 non-null  str  
 6   source_url        42902 non-null  str  
 7   rating            42902 non-null  int64
 8   helpful_count     42902 non-null  int64
 9   purchased_option  34411 non-null  str  
 10  has_image         42902 non-null  int64
 11  product_name      42902 non-null  str  
 12  product_id        42902 non-null  int64
dtypes: int64(5), str(8)
memory usage: 4.3 MB


In [88]:
df_alo_reviews.shape

(42902, 13)

In [89]:
(df_alo_reviews.isna().mean() * 100).sort_values(ascending=False)

purchased_option    19.791618
content              0.034963
review_id            0.000000
collect_date         0.000000
platform             0.000000
brand                0.000000
search_keyword       0.000000
source_url           0.000000
rating               0.000000
helpful_count        0.000000
has_image            0.000000
product_name         0.000000
product_id           0.000000
dtype: float64

In [158]:
df_alo_reviews['purchased_option']

0             L
1             M
2         Small
3           NaN
4            XS
          ...  
42897    Medium
42898       NaN
42899     Small
42900       NaN
42901      bone
Name: purchased_option, Length: 42902, dtype: str

## 네이버 블로그

In [69]:
df_naver_blog.info()

<class 'pandas.DataFrame'>
RangeIndex: 17533 entries, 0 to 17532
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   title        17527 non-null  str  
 1   link         17533 non-null  str  
 2   description  17523 non-null  str  
 3   postdate     17533 non-null  int64
dtypes: int64(1), str(3)
memory usage: 548.0 KB


In [70]:
df_naver_blog.shape

(17533, 4)

In [80]:
(df_naver_blog.isna().mean() * 100).sort_values(ascending=False)

description    0.057035
title          0.034221
link           0.000000
postdate       0.000000
dtype: float64

In [93]:
df_naver_blog['link']

0             https://blog.naver.com/ssujam/223578377725
1         https://blog.naver.com/minju1004b/224154715728
2            https://blog.naver.com/yoonkey/224253880817
3          https://blog.naver.com/hrelation/224162297856
4          https://blog.naver.com/qu14ux785/224226057832
                              ...                       
17528      https://blog.naver.com/cjstkahdk/224103294354
17529     https://blog.naver.com/thregitong/222577212470
17530    https://blog.naver.com/hearteffect/223464540115
17531      https://blog.naver.com/daily__oe/224261372710
17532     https://blog.naver.com/qikv796695/224261419613
Name: link, Length: 17533, dtype: str

In [4]:
df = pd.read_csv("data/blog.csv")

# 데이터 선택
texts = df["description"].dropna().tolist()

# 텍스트 정제
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^가-힣0-9a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# 명사 추출
def get_nouns(text):
    result = kiwi.analyze(text)[0][0]
    return [t.form for t in result if t.tag.startswith('N')]

# 실행
all_nouns = []

for t in texts:
    t = clean_text(t)
    nouns = get_nouns(t)
    all_nouns.extend(nouns)

# 빈도 분석
counter = Counter(all_nouns)

print(counter.most_common(30))

NameError: name 'Counter' is not defined